In [12]:
"""
Generic MySQL Agent with phidata (SELECT + DML/DDL w/ safety) — with VISIBLE step-by-step prints

Tested with:
  phidata==2.7.10
  openai>=1.0.0
  SQLAlchemy>=2.0
  PyMySQL>=1.1.0
  python-dotenv

Env:
  OPENAI_API_KEY=sk-...
  MYSQL_HOST=localhost
  MYSQL_PORT=3306
  MYSQL_USER=...
  MYSQL_PASSWORD=...
  MYSQL_DB=...
  # optional
  OPENAI_MODEL=gpt-4o-mini
  MYSQL_MAX_ROWS=200
  ALLOW_WRITE=false   # set to "true" to allow DML/DDL execution
"""

from dotenv import load_dotenv
load_dotenv()

import os
import re
from typing import Optional, List

# ---- phidata ---------------------------------------------------------------
from phi.agent import Agent
from phi.model.openai import OpenAIChat
from phi.tools import Toolkit

# ---- SQLAlchemy ------------------------------------------------------------
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.engine import Engine


# ====================== Helpers ============================================

_SELECT_START   = re.compile(r"(?is)^\s*select\b")
_WRITE_START    = re.compile(r"(?is)^\s*(insert|update|delete|merge|replace)\b")
_DDL_START      = re.compile(r"(?is)^\s*(create|alter|drop|truncate|rename)\b")
_LIMIT_PRESENT  = re.compile(r"(?is)\blimit\s+\d+(\s*,\s*\d+)?\b")

def _mysql_uri_from_env() -> str:
    host = os.getenv("MYSQL_HOST", "localhost")
    port = int(os.getenv("MYSQL_PORT", "3306"))
    user = os.getenv("MYSQL_USER")
    pwd  = os.getenv("MYSQL_PASSWORD")
    db   = os.getenv("MYSQL_DB")
    if not all([user, pwd, db]):
        raise RuntimeError("Set MYSQL_USER, MYSQL_PASSWORD, and MYSQL_DB in env")
    return f"mysql+pymysql://{user}:{pwd}@{host}:{port}/{db}"

def _get_engine() -> Engine:
    return create_engine(_mysql_uri_from_env(), pool_pre_ping=True)

def _render_schema(engine: Engine, tables: Optional[List[str]] = None) -> str:
    ins = inspect(engine)
    all_tables = sorted(ins.get_table_names())
    target_tables = all_tables if not tables else [t for t in tables if t in all_tables]
    if not target_tables:
        return "No tables found."

    lines: List[str] = []
    for t in target_tables:
        lines.append(f"# {t}")
        for col in ins.get_columns(t):
            colname = col.get("name")
            coltype = str(col.get("type"))
            nullable = col.get("nullable")
            default = col.get("default")
            lines.append(f"- {colname}: {coltype}, NULL={nullable}, DEFAULT={default}")
    head = "\n".join([f"- {t}" for t in all_tables[:50]])
    lines.append("\n# tables snapshot (truncated):\n" + head)
    return "\n".join(lines)

def _classify_sql(sql: str) -> str:
    s = (sql or "").strip()
    if not s:
        return "empty"
    if ";" in s:
        return "multi"
    if _SELECT_START.match(s):
        return "select"
    if _WRITE_START.match(s):
        return "write"
    if _DDL_START.match(s):
        return "ddl"
    return "other"  # SHOW/DESCRIBE/EXPLAIN/SET/etc.

def _maybe_inject_limit(sql: str, row_cap: int) -> str:
    if _SELECT_START.match(sql) and not _LIMIT_PRESENT.search(sql):
        return f"{sql} LIMIT {row_cap}"
    return sql


# ====================== Custom Toolkit (with explicit prints) ===============

class MySQLTools(Toolkit):
    """
    Tools:
      - mysql_schema(table_list: str="") -> str
      - mysql_run_sql(sql: str) -> str                    # generic single-statement executor
      - mysql_nl2sql(question: str) -> str               # NL -> SQL (any kind)
      - mysql_answer(question: str) -> str               # NL -> SQL -> RUN (respects ALLOW_WRITE)
    """
    def __init__(self, model: Optional[OpenAIChat] = None, row_cap: Optional[int] = None):
        super().__init__(name="mysql_tools")
        self.engine = _get_engine()
        self.model = model or OpenAIChat(id=os.getenv("OPENAI_MODEL", "gpt-4o-mini"), temperature=0)
        self.row_cap = row_cap or int(os.getenv("MYSQL_MAX_ROWS", "200"))
        self.allow_write = os.getenv("ALLOW_WRITE", "false").lower() == "true"

        self.register(self.mysql_schema)
        self.register(self.mysql_run_sql)
        self.register(self.mysql_nl2sql)
        self.register(self.mysql_answer)

    # ---------- Tool 1: Schema ---------------------------------------------
    def mysql_schema(self, table_list: str = "") -> str:
        print("[tool] mysql_schema called", flush=True)
        tables = [t.strip() for t in table_list.split(",") if t.strip()] if table_list else None
        try:
            return _render_schema(self.engine, tables)
        except Exception as e:
            return f"Schema error: {e}"

    # ---------- Tool 2: Generic SQL executor -------------------------------
    def mysql_run_sql(self, sql: str) -> str:
        print("[tool] mysql_run_sql called", flush=True)
        if not sql or not sql.strip():
            return "Please provide a SQL statement."
        sql = re.sub(r";.*$", "", sql, flags=re.S).strip()  # enforce single statement
        kind = _classify_sql(sql)

        if kind in ("write", "ddl") and not self.allow_write:
            return ("Write/DDL blocked by safety policy. "
                    "Set ALLOW_WRITE=true to enable. The statement was classified as "
                    f"'{kind}'.\nSQL (blocked): {sql}")

        if kind == "select":
            sql = _maybe_inject_limit(sql, self.row_cap)

        print(f"[exec] Kind={kind} | SQL={sql}", flush=True)
        try:
            with self.engine.begin() as conn:
                result = conn.execute(text(sql))
                if kind == "select":
                    rows = result.fetchmany(self.row_cap + 1)
                    headers = list(result.keys())
                    truncated = len(rows) > self.row_cap
                    rows = rows[:self.row_cap]
                    out = []
                    header_line = " | ".join(map(str, headers))
                    out.append(header_line)
                    out.append("-" * max(3, len(header_line)))
                    for r in rows:
                        out.append(" | ".join("" if v is None else str(v) for v in r))
                    if truncated:
                        out.append(f"...(truncated at {self.row_cap} rows)")
                    return "\n".join(out)
                else:
                    try:
                        rowcount = result.rowcount
                    except Exception:
                        rowcount = None
                    return f"OK ({kind}). Rows affected: {rowcount}."
        except Exception as e:
            return f"Execution error ({kind}): {e}"

    # ---------- Tool 3: NL -> SQL (any) ------------------------------------
    def mysql_nl2sql(self, question: str) -> str:
        print("[tool] mysql_nl2sql called", flush=True)
        if not question or not question.strip():
            return "Please provide a question."

        # include schema to ground column/table names
        try:
            schema_text = _render_schema(self.engine, None)
        except Exception as e:
            return f"Failed to load schema: {e}"

        system = (
            "You are a senior data engineer. Produce ONE valid MySQL statement that "
            "answers the user's request using only existing tables/columns from the schema. "
            "Be concise and correct. No comments/prose. No trailing ';'."
        )
        user = f"Database schema:\n{schema_text}\n\nRequest:\n{question}\n\nMySQL:"

        try:
            sql = (self.model.response(messages=[{"role": "system", "content": system},
                                                 {"role": "user", "content": user}]) or "").strip()
        except Exception as e:
            return f"NL2SQL error: {e}"

        sql = re.sub(r";.*$", "", sql, flags=re.S).strip()
        print(f"[nl2sql] SQL generated: {sql}", flush=True)
        if not sql:
            return "Failed to generate SQL. Try rephrasing."
        return sql

    # ---------- Tool 4: NL -> SQL -> RUN -----------------------------------
    def mysql_answer(self, question: str) -> str:
        print("[tool] mysql_answer called", flush=True)
        sql = self.mysql_nl2sql(question)
        if not sql or "error" in sql.lower() or "failed" in sql.lower():
            return sql
        result = self.mysql_run_sql(sql)
        return result


# ====================== Agent builder =======================================

def build_agent(model_name: Optional[str] = None) -> Agent:
    """
    Agent policy:
      - If user provides explicit SQL, call mysql_run_sql.
      - If user asks in NL, call mysql_answer.
      - Use mysql_schema for structure questions.
    """
    llm_id = model_name or os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    model = OpenAIChat(id=llm_id, temperature=0)

    tools = [MySQLTools(model=model)]

    instructions = (
        "You are a MySQL operations and analytics assistant.\n"
        "- If the user provides explicit SQL, call `mysql_run_sql` with it.\n"
        "- If the user asks a question in natural language, call `mysql_answer`.\n"
        "- If they ask about tables/columns, call `mysql_schema` (optionally with a table list).\n"
        "- Keep answers concise. Always return the DB result or a clear error.\n"
        "- Assume single-statement execution. Do not fabricate results."
    )

    return Agent(
        model=model,
        tools=tools,
        show_tool_calls=True,
        markdown=True,
        instructions=instructions,
    )


# ====================== Trace printer for Agent.run =========================

def pretty_print_run(run_obj) -> None:
    """Print every message & tool step from a phidata Agent run."""
    print("\n=== Agent Run Trace ===", flush=True)
    msgs = getattr(run_obj, "messages", None)
    if not msgs:
        print(getattr(run_obj, "content", ""), flush=True)
        print("=== End Trace ===\n", flush=True)
        return
    for m in msgs:
        role = getattr(m, "role", "") or getattr(m, "type", "")
        content = getattr(m, "content", "")
        name = getattr(m, "name", "")
        if role == "tool":
            print(f"[tool:{name}] {content}", flush=True)
        else:
            label = role or "assistant"
            print(f"[{label}] {content}", flush=True)
    print("=== End Trace ===\n", flush=True)


# ====================== Minimal demo ========================================

if __name__ == "__main__":
    import sys, time
    print("[demo] Generic MySQL Agent starting…", flush=True)
    try:
        engine = _get_engine()
        with engine.connect() as c:
            c.execute(text("SELECT 1"))
        print("[demo] DB connection OK.", flush=True)
    except Exception as e:
        print(f"[demo] DB connection failed: {e}", flush=True)
        raise SystemExit(1)

    # Build both: agent (for chat-style) and a direct tools instance (deterministic fallback)
    agent = build_agent()
    tools: MySQLTools = agent.tools[0]  # type: ignore

    start = time.time()

    # Choose your prompt here:
    # user_input = " ".join(sys.argv[1:]) or "Show first 5 rows from datasets"
    #user_input = "provide me description & counts of all columns of the table datasets and its significance"
    #user_input="provide me counts of each table"
    #user_input="provide me description of tags table"
    #user_input="provide me description of params table"
    #user_input="provide me best metrics and its value for experiment id 1"
    #user_input="provide me count of runs for experiment id 1"
    #user_input="what are the distinct metrics and its maximum values for experiment id 1"
    #user_input=" provide me  distinct metric and its value each experiment"
    user_input="provide me count of params for each run and for each experiment"
    
    print(f"[demo] Prompt: {user_input}\n", flush=True)

    # ---- (A) Deterministic fallback: ALWAYS prints intermediate steps ----
    print("[demo] Deterministic path: NL -> SQL -> EXECUTE", flush=True)
    sql_generated = tools.mysql_nl2sql(user_input)
    print(f"[demo] Generated SQL:\n{sql_generated}\n", flush=True)
    result = tools.mysql_run_sql(sql_generated if sql_generated else "SELECT 1")
    print(f"[demo] Result:\n{result}\n", flush=True)

    # ---- (B) Agent path: show a full trace instead of spinner-only output ----
    print("[demo] Agent path: run() + trace (instead of print_response())", flush=True)
    run = agent.run(user_input)
    pretty_print_run(run)
    print("[demo] Agent final answer:\n" + (getattr(run, "content", "") or ""), flush=True)

    print(f"\n[demo] Done in {time.time()-start:.2f}s", flush=True)


[demo] Generic MySQL Agent starting…
[demo] DB connection OK.
[demo] Prompt: provide me count of params for each run and for each experiment

[demo] Deterministic path: NL -> SQL -> EXECUTE
[tool] mysql_nl2sql called
[demo] Generated SQL:
NL2SQL error: 'dict' object has no attribute 'log'

[tool] mysql_run_sql called
[exec] Kind=other | SQL=NL2SQL error: 'dict' object has no attribute 'log'
[demo] Result:
Execution error (other): (pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MariaDB server version for the right syntax to use near 'NL2SQL error: 'dict' object has no attribute 'log'' at line 1")
[SQL: NL2SQL error: 'dict' object has no attribute 'log']
(Background on this error at: https://sqlalche.me/e/20/f405)

[demo] Agent path: run() + trace (instead of print_response())
[tool] mysql_run_sql called
[exec] Kind=select | SQL=SELECT experiment_id, COUNT(param_id) AS param_count FROM params GROUP BY experiment_id LI